# Heat Equation Inverse with DeepXDE
I attempt to solve an inverse problem of the Heat Equation with DeepXDE.

In [1]:
import deepxde as dde
import numpy as np
import torch

No backend selected.
Finding available backend...
Found pytorch
Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting the default backend to "pytorch". You can change it in the ~/.deepxde/config.json file or export the DDE_BACKEND environment variable. Valid options are: tensorflow.compat.v1, tensorflow, pytorch, jax, paddle (all lowercase)


In [2]:
# define the domain
geom = dde.geometry.Interval(0.0, 1.0) # so x is from 0 to 1
timedomain = dde.geometry.TimeDomain(0.0, 1.0) # so t is from 0 to 1
geomtime = dde.geometry.GeometryXTime(geom, timedomain) # space time domain

In [9]:
# define the PDE

# unknown parameter alpha
alpha = dde.Variable(0.5)

# define the PDE residual
def heat_pde(x, u):
    u_t = dde.grad.jacobian(u, x, i=0, j=1)  # du/dt
    u_xx = dde.grad.hessian(u, x, i=0, j=0)  # d2u/dx2
    return u_t - alpha * u_xx

# define the initial condition
def initial_condition(x):
    return np.sin(np.pi * x[:, 0:1])  # u(x,0) = sin(pi*x)

ic = dde.icbc.IC(
    geomtime,
    initial_condition,
    lambda _, on_initial: on_initial
)

# boundary conditions
bc_left = dde.icbc.DirichletBC(
    geomtime,
    lambda x: 0.0,
    lambda x, on_boundary: on_boundary and np.isclose(x[0], 0.0)
)

bc_right = dde.icbc.DirichletBC(
    geomtime,
    lambda x: 0.0,
    lambda x, on_boundary: on_boundary and np.isclose(x[0], 1.0)
)



In [10]:
# Observational data (optional)
# x_obs: shape (N, 2)
# u_obs: shape (N, 1)
# observe_u = dde.icbc.PointSetBC(x_obs, u_obs)

In [12]:
# data object
data = dde.data.TimePDE(
    geomtime,
    heat_pde,
    [ic, bc_left, bc_right],
    num_domain=10000,
    num_boundary=200,
    num_initial=200
)

# neural network
net = dde.nn.FNN(
    [2] + [50] * 3 + [1],
    activation="tanh",
    kernel_initializer="Glorot normal"
)

In [14]:
model = dde.Model(data, net)

model.compile(
    optimizer="adam",
    lr=1e-3,
    external_trainable_variables=[alpha]
)

Compiling model...
'compile' took 1.325704 s



In [ ]:
model.train(epochs=5000)

print("Learned alpha:", alpha.item())

Learned alpha: 0.0695977509021759
